# 架构

In [ ]:
# gan_arch.py  ——  只做架构定义 + 维度打印，不含训练
import torch
import torch.nn as nn


# =====================================================================
# 1) Generator：z ~ N(0, I) ∈ R^100  -->  x ∈ R^784 (28x28)
#    论文设定：隐层 ReLU，输出层 Sigmoid（像素归一化到 [0,1]）
# =====================================================================
class Generator(nn.Module):
    def __init__(self, z_dim: int = 100, img_dim: int = 784, hidden: int = 256):
        super().__init__()
        self.z_dim = z_dim
        self.img_dim = img_dim
        self.net = nn.Sequential(
            nn.Linear(z_dim,        hidden),        # 100 -> 256
            nn.ReLU(inplace=True),
            nn.Linear(hidden,       hidden * 2),    # 256 -> 512
            nn.ReLU(inplace=True),
            nn.Linear(hidden * 2,   hidden * 4),    # 512 -> 1024
            nn.ReLU(inplace=True),
            nn.Linear(hidden * 4,   img_dim),       # 1024 -> 784
            nn.Sigmoid(),                           # 输出概率图 [0,1]
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)                          # (B, z_dim) -> (B, 784)


# =====================================================================
# 2) Discriminator：x ∈ R^784 -->  标量概率
#    论文用 maxout + dropout，实践中普遍用 LeakyReLU + Dropout
# =====================================================================
class Discriminator(nn.Module):
    def __init__(self, img_dim: int = 784, hidden: int = 256, dropout: float = 0.3):
        super().__init__()
        self.img_dim = img_dim
        self.net = nn.Sequential(
            nn.Linear(img_dim,      hidden * 4),    # 784 -> 1024
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden * 4,   hidden * 2),    # 1024 -> 512
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2,   hidden),        # 512 -> 256
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden,       1),             # 256 -> 1
            nn.Sigmoid(),                           # 真/假概率
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)                          # (B, 784) -> (B, 1)


# =====================================================================
# 3) 工具：用 forward hook 自动打印每一层的输入/输出维度
# =====================================================================
@torch.no_grad()
def print_shapes(model: nn.Module, x: torch.Tensor, name: str = "Model"):
    print(f"\n{'=' * 62}\n>>> {name}\n{'=' * 62}")
    print(f"{'layer':<28}{'in shape':<18}{'out shape'}")
    print("-" * 62)

    handles = []

    def make_hook(idx):
        def hook(module, inp, out):
            cls = module.__class__.__name__
            if isinstance(module, nn.Linear):
                cls = f"Linear({module.in_features}->{module.out_features})"
            in_shape = tuple(inp[0].shape) if isinstance(inp, tuple) else tuple(inp.shape)
            out_shape = tuple(out.shape) if isinstance(out, torch.Tensor) else tuple(out[0].shape)
            print(f"[{idx:02d}] {cls:<24}{str(in_shape):<18}{str(out_shape)}")
        return hook

    # 只对叶子模块（真正做运算的层）挂 hook
    leaf_idx = 0
    for m in model.modules():
        if m is model:
            continue
        if len(list(m.children())) == 0:
            handles.append(m.register_forward_hook(make_hook(leaf_idx)))
            leaf_idx += 1

    model.eval()
    print(f"{'Input':<32}{tuple(x.shape)}")
    y = model(x)
    for h in handles:
        h.remove()

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("-" * 62)
    print(f"Output: {tuple(y.shape)}   |  trainable params: {n_params:,}")
    return y


# =====================================================================
# 4) 跑一遍看维度
# =====================================================================
if __name__ == "__main__":
    torch.manual_seed(0)

    Z_DIM, IMG_DIM, BATCH = 100, 28 * 28, 8

    G = Generator(z_dim=Z_DIM, img_dim=IMG_DIM)
    D = Discriminator(img_dim=IMG_DIM)

    z    = torch.randn(BATCH, Z_DIM)     # 噪声 z ~ N(0, I)
    real = torch.rand(BATCH, IMG_DIM)    # 假装是 MNIST 像素（真实训练时来自 dataloader）

    fake = print_shapes(G, z,    "Generator  G(z)")
    print_shapes(D, fake.detach(), "Discriminator  D(fake)")
    print_shapes(D, real,          "Discriminator  D(real)")


# 实战1

数据是 28×28 的 MNIST（训练 60,000，测试 10,000）。`DATASET` 改成 `"FashionMNIST"` 可换成服装图。`R1_GAMMA = 10` 用来把 `D(real)` 和 `D(fake)` 拉到 0.5 附近。


In [ ]:
# 实战：自包含，不依赖上面的架构 cell
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils as vutils

# ---------------------------------------------------------------------
# 配置
# ---------------------------------------------------------------------
DATA_DIR = Path("./data")
DATASET = "MNIST"          # 或 "FashionMNIST"，同样是 28x28
Z_DIM = 100
IMG_DIM = 28 * 28
BATCH_SIZE = 128
EPOCHS = 20                # GPU 上大约两三分钟；冒烟测试可改成 2
LR = 2e-4
BETAS = (0.5, 0.999)       # 原论文用 momentum SGD；这组 Adam 参数在 MLP-GAN 上更稳
K_D = 1                    # 每更新一次 G，D 更新几步（论文 Algorithm 1）
R1_GAMMA = 10.0            # 真实样本上的梯度惩罚，把 D(real)/D(fake) 拉向 0.5
SEED = 0
N_SHOW = 64                # 固定噪声，观察同一批 z 随训练的变化
OUT_DIR = Path("./runs/mnist_gan")

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("device:", device)


# ---------------------------------------------------------------------
# 网络：与上面的架构一致，写在这里是为了这一格能单独运行
# Generator：z ~ N(0, I) -> x ∈ [0, 1]^784，隐层 ReLU，输出 Sigmoid
# Discriminator：x -> 真假概率，LeakyReLU + Dropout
# ---------------------------------------------------------------------
class Generator(nn.Module):
    def __init__(self, z_dim: int = 100, img_dim: int = 784, hidden: int = 256):
        super().__init__()
        self.z_dim = z_dim
        self.img_dim = img_dim
        self.net = nn.Sequential(
            nn.Linear(z_dim, hidden), # 100 -> 256
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden * 2), # 256 -> 512
            nn.ReLU(inplace=True),
            nn.Linear(hidden * 2, hidden * 4), # 512 -> 1024
            nn.ReLU(inplace=True),
            nn.Linear(hidden * 4, img_dim), 
            nn.Sigmoid(), #  压到 [0, 1]（MNIST 像素）
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, img_dim: int = 784, hidden: int = 256, dropout: float = 0.3):
        super().__init__()
        self.img_dim = img_dim
        self.net = nn.Sequential(
            nn.Linear(img_dim, hidden * 4),
            nn.LeakyReLU(0.2, inplace=True), # 负半轴斜率 0.2。
            # D 用 LeakyReLU 而不是 ReLU，因为 D 学到"一刀切"决策时若 ReLU 全负则梯度消失，
            # G 完全拿不到信号。这是 GAN 训练不稳定的常见坑。 
            nn.Dropout(dropout),
            nn.Linear(hidden * 4, hidden * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, hidden),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1), # 形状 (B, 1)，值域 (-∞, +∞) ← 这是 logit
            nn.Sigmoid(), # 形状 (B, 1)，值域 (0, 1)   ← 这是概率
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def d_logits(model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    """Sigmoid 前的 logit。概率饱和后 BCE(sigmoid) 梯度为 0，logit 上仍有梯度。"""
    layers = list(model.net.children())
    if not isinstance(layers[-1], nn.Sigmoid):
        raise RuntimeError("Discriminator 最后一层应为 Sigmoid")
    h = x
    for layer in layers[:-1]:
        h = layer(h)
    return h


# ---------------------------------------------------------------------
# 数据
# ---------------------------------------------------------------------
if DATASET == "MNIST":
    datasets.MNIST.mirrors = [
        "https://storage.googleapis.com/cvdf-datasets/mnist/",
        "https://ossci-datasets.s3.amazonaws.com/mnist/",
    ]

train_set = getattr(datasets, DATASET)(
    DATA_DIR, train=True, download=True, transform=transforms.ToTensor()
)
test_set = getattr(datasets, DATASET)(
    DATA_DIR, train=False, download=True, transform=transforms.ToTensor()
)
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    test_set,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
print(f"{DATASET}  train {len(train_set)}   test {len(test_set)}   steps/epoch {len(train_loader)}")


# ---------------------------------------------------------------------
# 训练
# 判别器：最大化 log D(x) + log(1 - D(G(z)))
# 生成器：论文第 3 节 non-saturating 目标，最大化 log D(G(z))
# R1 惩罚真实图像上 logit 的梯度范数，均衡点仍是 D = 0.5
# ---------------------------------------------------------------------
def init_linear(module):
    if isinstance(module, nn.Linear):
        nn.init.xavier_normal_(module.weight)
        nn.init.zeros_(module.bias)


G = Generator(z_dim=Z_DIM, img_dim=IMG_DIM).to(device)
D = Discriminator(img_dim=IMG_DIM).to(device)
G.apply(init_linear)
D.apply(init_linear)

opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)
bce_logits = nn.BCEWithLogitsLoss() # logit 版本交叉熵
fixed_z = torch.randn(N_SHOW, Z_DIM, device=device)
history = {"d": [], "g": [], "d_real": [], "d_fake": []}
snapshots = {}


def discriminator_step(real):
    real = real.detach().requires_grad_(True)
    fake = G(torch.randn(real.size(0), Z_DIM, device=device)).detach()
    logit_real = d_logits(D, real)
    logit_fake = d_logits(D, fake)
    loss = bce_logits(logit_real, torch.ones_like(logit_real))
    loss = loss + bce_logits(logit_fake, torch.zeros_like(logit_fake))
    grad = torch.autograd.grad(logit_real.sum(), real, create_graph=True)[0]
    loss = loss + (R1_GAMMA / 2) * grad.flatten(1).pow(2).sum(1).mean()
    opt_D.zero_grad(set_to_none=True)
    loss.backward()
    opt_D.step()
    return (
        loss.item(),
        torch.sigmoid(logit_real.detach()).mean().item(),
        torch.sigmoid(logit_fake.detach()).mean().item(),
    )


def generator_step(batch_size):
    logit_fake = d_logits(D, G(torch.randn(batch_size, Z_DIM, device=device)))
    loss = bce_logits(logit_fake, torch.ones_like(logit_fake))
    opt_G.zero_grad(set_to_none=True)
    loss.backward()
    opt_G.step()
    return loss.item()


for epoch in range(1, EPOCHS + 1):
    G.train()
    D.train()
    sum_d = sum_g = sum_real = sum_fake = 0.0
    n_batches = 0
    for real, _ in train_loader:
        real = real.view(real.size(0), -1).to(device, non_blocking=True)
        for _ in range(K_D):
            loss_d, score_real, score_fake = discriminator_step(real)
            sum_d += loss_d
            sum_real += score_real
            sum_fake += score_fake
        sum_g += generator_step(real.size(0))
        n_batches += 1

    history["d"].append(sum_d / (n_batches * K_D))
    history["g"].append(sum_g / n_batches)
    history["d_real"].append(sum_real / (n_batches * K_D))
    history["d_fake"].append(sum_fake / (n_batches * K_D))

    G.eval()
    with torch.no_grad():
        snapshots[epoch] = G(fixed_z).view(-1, 1, 28, 28).cpu()

    print(
        f"epoch {epoch:02d}/{EPOCHS}  "
        f"D {history['d'][-1]:.3f}  G {history['g'][-1]:.3f}  "
        f"D(real) {history['d_real'][-1]:.3f}  D(fake) {history['d_fake'][-1]:.3f}"
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "G": G.state_dict(),
        "D": D.state_dict(),
        "z_dim": Z_DIM,
        "history": history,
        "dataset": DATASET,
        "r1_gamma": R1_GAMMA,
    },
    OUT_DIR / "ckpt.pt",
)
print("saved", OUT_DIR / "ckpt.pt")


# ---------------------------------------------------------------------
# 绘图
# ---------------------------------------------------------------------
epochs = range(1, len(history["d"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
axes[0].plot(epochs, history["d"], label="D loss")
axes[0].plot(epochs, history["g"], label="G loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("BCE")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history["d_real"], label="D(real)")
axes[1].plot(epochs, history["d_fake"], label="D(fake)")
axes[1].axhline(0.5, color="gray", ls="--", lw=1)
axes[1].set_xlabel("epoch")
axes[1].set_ylim(0, 1)
axes[1].set_title("Discriminator score")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "loss.png", dpi=140)
plt.show()

show_epochs = sorted({1, max(1, EPOCHS // 2), EPOCHS})
fig, axes = plt.subplots(1, len(show_epochs), figsize=(4.2 * len(show_epochs), 4.2))
if len(show_epochs) == 1:
    axes = [axes]
for ax, epoch in zip(axes, show_epochs):
    grid = vutils.make_grid(snapshots[epoch], nrow=8, padding=2, pad_value=1.0)
    img = grid.permute(1, 2, 0).numpy()
    ax.imshow(img.squeeze(), cmap="gray" if img.shape[-1] == 1 else None)
    ax.set_title(f"fixed z, epoch {epoch}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "samples_progress.png", dpi=140)
plt.show()


# ---------------------------------------------------------------------
# 测试：官方 test split 上的判别器打分和论文中的价值函数
# V(D, G) = E[log D(x)] + E[log(1 - D(G(z)))]
# 纳什均衡时两边都接近 0.5，V = -log 4 ≈ -1.386
# ---------------------------------------------------------------------
G.eval()
D.eval()
real_scores = []
fake_scores = []
with torch.no_grad():
    for real, _ in test_loader:
        real = real.view(real.size(0), -1).to(device, non_blocking=True)
        real_scores.append(D(real).cpu())
        fake_scores.append(D(G(torch.randn(real.size(0), Z_DIM, device=device))).cpu())

real_scores = torch.cat(real_scores).squeeze()
fake_scores = torch.cat(fake_scores).squeeze()
eps = 1e-6
term_real = torch.log(real_scores.clamp(min=eps)).mean()
term_fake = torch.log((1 - fake_scores).clamp(min=eps)).mean()
print(f"test images: {real_scores.numel()}")
print(f"D(real)  mean {real_scores.mean():.3f}  std {real_scores.std():.3f}")
print(f"D(fake)  mean {fake_scores.mean():.3f}  std {fake_scores.std():.3f}")
print(f"E log D(x) = {term_real.item():.3f}")
print(f"E log(1 - D(G(z))) = {term_fake.item():.3f}")
print(f"V = {term_real.item() + term_fake.item():.3f}    (-log 4 = {-torch.log(torch.tensor(4.)).item():.3f})")

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6))
axes[0].hist(real_scores.numpy(), bins=30, range=(0, 1), alpha=0.85, label="real test")
axes[0].hist(fake_scores.numpy(), bins=30, range=(0, 1), alpha=0.7, label="G(z)")
axes[0].set_xlabel("D(x)")
axes[0].set_ylabel("count")
axes[0].set_title("Discriminator on the test split")
axes[0].legend()

real_batch, _ = next(iter(test_loader))
with torch.no_grad():
    fake_batch = G(torch.randn(32, Z_DIM, device=device)).view(-1, 1, 28, 28).cpu()
pair = torch.cat([real_batch[:32], fake_batch], dim=0)
grid = vutils.make_grid(pair, nrow=8, padding=2, pad_value=1.0)
img = grid.permute(1, 2, 0).numpy()
axes[1].imshow(img.squeeze(), cmap="gray" if img.shape[-1] == 1 else None)
axes[1].set_title("rows 1-4 real test, rows 5-8 generated")
axes[1].axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "test_eval.png", dpi=140)
plt.show()


# 实战2

数据用 **CIFAR-10**：32×32 RGB，10 类，训练 50,000 张、测试 10,000 张。2014 年 GAN 论文也用过它。彩色图改成卷积生成器和判别器（DCGAN），像素缩放到 `[-1, 1]`，对应输出层的 Tanh。`R1_GAMMA = 1` 用来把两边的分数拉向 0.5。


In [ ]:
# 实战2：自包含，不依赖上面的架构 cell 和实战1
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils as vutils

# ---------------------------------------------------------------------
# 配置
# CIFAR-10：32x32 彩色图，10 类，训练 50,000，测试 10,000，压缩包约 170MB。
# 2014 年 GAN 论文也在 CIFAR-10 上做过实验。彩色图用卷积生成器 / 判别器
# （DCGAN 结构），保留空间结构；输出 Tanh，像素缩放到 [-1, 1]。
# ---------------------------------------------------------------------
DATA_DIR = Path("./data")
Z_DIM = 100
IMG_SIZE = 32
IMG_CH = 3
NGF = 64 # 判别器过滤器数量
NDF = 64 # 生成器过滤器数量
BATCH_SIZE = 128
EPOCHS = 20
LR = 2e-4
BETAS = (0.5, 0.999)
K_D = 1
R1_GAMMA = 1.0             # 32x32 卷积网上比 MNIST 的 MLP 小一档，仍把分数拉向 0.5
SEED = 0
N_SHOW = 64
OUT_DIR = Path("./runs/cifar10_gan")
CLASSES = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
)

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
print("device:", device)


# ---------------------------------------------------------------------
# 网络
# G: z (B, 100) -> (B, 3, 32, 32)，范围 [-1, 1]
# D: (B, 3, 32, 32) -> logit (B, 1)，最后一层不加 Sigmoid
# ---------------------------------------------------------------------
class Generator(nn.Module):
    def __init__(self, z_dim: int = 100, ngf: int = 64, nc: int = 3):
        super().__init__()
        self.z_dim = z_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, ngf * 4, 4, 1, 0, bias=False),   # 4x4
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False), # 8x8
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),     # 16x16
            nn.BatchNorm2d(ngf),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(ngf, nc, 4, 2, 1, bias=False),          # 32x32
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z.view(z.size(0), z.size(1), 1, 1))


class Discriminator(nn.Module):
    def __init__(self, ndf: int = 64, nc: int = 3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),                   # 16x16
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),              # 8x8
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),          # 4x4
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),                # 1x1 logit
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(x.size(0), 1)


def init_dcgan(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, 0.0, 0.02)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.normal_(module.weight, 1.0, 0.02)
        nn.init.zeros_(module.bias)


def to_unit(imgs: torch.Tensor) -> torch.Tensor:
    """[-1, 1] -> [0, 1]，只用于显示。"""
    return (imgs.clamp(-1, 1) + 1) / 2


# ---------------------------------------------------------------------
# 数据
# ---------------------------------------------------------------------
datasets.CIFAR10.url = "https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz"
tfm = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
train_set = datasets.CIFAR10(
    DATA_DIR, train=True, download=True, transform=transforms.Compose([transforms.ToTensor(), tfm])
)
test_set = datasets.CIFAR10(
    DATA_DIR, train=False, download=True, transform=transforms.Compose([transforms.ToTensor(), tfm])
)
train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    test_set,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
print(f"CIFAR-10  train {len(train_set)}   test {len(test_set)}   steps/epoch {len(train_loader)}")

real_preview, labels = next(iter(train_loader))
grid = vutils.make_grid(to_unit(real_preview[:N_SHOW]), nrow=8, padding=2, pad_value=1.0)
plt.figure(figsize=(5.2, 5.2))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title("CIFAR-10 training samples")
plt.axis("off")
plt.show()
print("first labels:", [CLASSES[i] for i in labels[:8].tolist()])


# ---------------------------------------------------------------------
# 训练
# 判别器：最大化 log D(x) + log(1 - D(G(z)))，用 logit 上的 BCE
# 生成器：non-saturating，最大化 log D(G(z))
# R1 惩罚真实图像上 logit 的梯度范数
# ---------------------------------------------------------------------
G = Generator(z_dim=Z_DIM, ngf=NGF, nc=IMG_CH).to(device)
D = Discriminator(ndf=NDF, nc=IMG_CH).to(device)
G.apply(init_dcgan)
D.apply(init_dcgan)

opt_G = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
opt_D = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)
bce_logits = nn.BCEWithLogitsLoss()
fixed_z = torch.randn(N_SHOW, Z_DIM, device=device)
history = {"d": [], "g": [], "d_real": [], "d_fake": []}
snapshots = {}


def discriminator_step(real):
    real = real.detach().requires_grad_(True)
    fake = G(torch.randn(real.size(0), Z_DIM, device=device)).detach()
    logit_real = D(real)
    logit_fake = D(fake)
    loss = bce_logits(logit_real, torch.ones_like(logit_real))
    loss = loss + bce_logits(logit_fake, torch.zeros_like(logit_fake))
    # loss = BCE(logit_real, 1) + BCE(logit_fake, 0)
    # 就是论文的 D 目标 max E[log D(x)] + E[log(1 - D(G(z)))]
    grad = torch.autograd.grad(logit_real.sum(), real, create_graph=True)[0]
    loss = loss + (R1_GAMMA / 2) * grad.flatten(1).pow(2).sum(1).mean()
    opt_D.zero_grad(set_to_none=True)
    loss.backward()
    opt_D.step()
    return (
        loss.item(),
        torch.sigmoid(logit_real.detach()).mean().item(),
        torch.sigmoid(logit_fake.detach()).mean().item(),
    )


def generator_step(batch_size):
    logit_fake = D(G(torch.randn(batch_size, Z_DIM, device=device)))
    loss = bce_logits(logit_fake, torch.ones_like(logit_fake))
    # 等价于 max log D(G(z))，即 min -log D(G(z))
    # 早期 D(G(z)) ≈ 0，-log D ≈ -log(0) → +∞，loss 巨大
    # 梯度巨大，G 有强信号知道"假样本被识破了，赶紧改"
    opt_G.zero_grad(set_to_none=True)
    loss.backward()
    opt_G.step()
    return loss.item()


for epoch in range(1, EPOCHS + 1):
    G.train()
    D.train()
    sum_d = sum_g = sum_real = sum_fake = 0.0
    n_batches = 0
    for real, _ in train_loader:
        real = real.to(device, non_blocking=True)
        for _ in range(K_D):
            loss_d, score_real, score_fake = discriminator_step(real)
            sum_d += loss_d
            sum_real += score_real
            sum_fake += score_fake
        sum_g += generator_step(real.size(0))
        n_batches += 1

    history["d"].append(sum_d / (n_batches * K_D))
    history["g"].append(sum_g / n_batches)
    history["d_real"].append(sum_real / (n_batches * K_D))
    history["d_fake"].append(sum_fake / (n_batches * K_D))

    G.eval()
    with torch.no_grad():
        snapshots[epoch] = G(fixed_z).cpu()

    print(
        f"epoch {epoch:02d}/{EPOCHS}  "
        f"D {history['d'][-1]:.3f}  G {history['g'][-1]:.3f}  "
        f"D(real) {history['d_real'][-1]:.3f}  D(fake) {history['d_fake'][-1]:.3f}"
    )

OUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "G": G.state_dict(),
        "D": D.state_dict(),
        "z_dim": Z_DIM,
        "history": history,
        "r1_gamma": R1_GAMMA,
    },
    OUT_DIR / "ckpt.pt",
)
print("saved", OUT_DIR / "ckpt.pt")


# ---------------------------------------------------------------------
# 绘图
# ---------------------------------------------------------------------
epochs = range(1, len(history["d"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.4))
axes[0].plot(epochs, history["d"], label="D loss")
axes[0].plot(epochs, history["g"], label="G loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("BCE")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history["d_real"], label="D(real)")
axes[1].plot(epochs, history["d_fake"], label="D(fake)")
axes[1].axhline(0.5, color="gray", ls="--", lw=1)
axes[1].set_xlabel("epoch")
axes[1].set_ylim(0, 1)
axes[1].set_title("Discriminator score")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "loss.png", dpi=140)
plt.show()

show_epochs = sorted({1, max(1, EPOCHS // 2), EPOCHS})
fig, axes = plt.subplots(1, len(show_epochs), figsize=(4.2 * len(show_epochs), 4.2))
if len(show_epochs) == 1:
    axes = [axes]
for ax, epoch in zip(axes, show_epochs):
    grid = vutils.make_grid(to_unit(snapshots[epoch]), nrow=8, padding=2, pad_value=1.0)
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(f"fixed z, epoch {epoch}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "samples_progress.png", dpi=140)
plt.show()


# ---------------------------------------------------------------------
# 测试：官方 test split 上的判别器打分和论文中的价值函数
# V = E[log D(x)] + E[log(1 - D(G(z)))]，均衡时约为 -log 4 ≈ -1.386
# ---------------------------------------------------------------------
G.eval()
D.eval()
real_scores = []
fake_scores = []
with torch.no_grad():
    for real, _ in test_loader:
        real = real.to(device, non_blocking=True)
        real_scores.append(torch.sigmoid(D(real)).cpu())
        fake_scores.append(torch.sigmoid(D(G(torch.randn(real.size(0), Z_DIM, device=device)))).cpu())

real_scores = torch.cat(real_scores).squeeze()
fake_scores = torch.cat(fake_scores).squeeze()
eps = 1e-6
term_real = torch.log(real_scores.clamp(min=eps)).mean()
term_fake = torch.log((1 - fake_scores).clamp(min=eps)).mean()
print(f"test images: {real_scores.numel()}")
print(f"D(real)  mean {real_scores.mean():.3f}  std {real_scores.std():.3f}")
print(f"D(fake)  mean {fake_scores.mean():.3f}  std {fake_scores.std():.3f}")
print(f"E log D(x) = {term_real.item():.3f}")
print(f"E log(1 - D(G(z))) = {term_fake.item():.3f}")
print(f"V = {term_real.item() + term_fake.item():.3f}    (-log 4 = {-torch.log(torch.tensor(4.)).item():.3f})")

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6))
axes[0].hist(real_scores.numpy(), bins=30, range=(0, 1), alpha=0.85, label="real test")
axes[0].hist(fake_scores.numpy(), bins=30, range=(0, 1), alpha=0.7, label="G(z)")
axes[0].set_xlabel("D(x)")
axes[0].set_ylabel("count")
axes[0].set_title("Discriminator on the test split")
axes[0].legend()

real_batch, _ = next(iter(test_loader))
with torch.no_grad():
    fake_batch = G(torch.randn(32, Z_DIM, device=device)).cpu()
pair = to_unit(torch.cat([real_batch[:32], fake_batch], dim=0))
grid = vutils.make_grid(pair, nrow=8, padding=2, pad_value=1.0)
axes[1].imshow(grid.permute(1, 2, 0).numpy())
axes[1].set_title("rows 1-4 real test, rows 5-8 generated")
axes[1].axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "test_eval.png", dpi=140)
plt.show()


# R1 惩罚是"怎么做到的"

---

## 一、核心回答（先给结论）

R1 是通过**对"梯度"再做一次求导（二阶梯度）**实现的：

```
普通训练：  更新 D 的参数 θ  →  让 D(x) 接近标签 y
R1 惩罚：   更新 D 的参数 θ  →  让 ∂D(x)/∂x 接近 0
            ↑
      这是"对导数的导数"，即二阶信息
```

**技术上靠 `torch.autograd.grad(..., create_graph=True)` + `loss.backward()` 完成**。下面拆开讲。

---

## 二、第一层：数学上"梯度小"到底意味着什么

### 2.1 梯度 = 局部斜率

对一元函数 `f(x)`，`f'(x₀)` 是 `x₀` 处的**切线斜率**。

- `f'(x₀) = 5`：x 稍微变一点，f 剧烈变化 → 函数"陡峭"
- `f'(x₀) = 0.01`：x 怎么变，f 几乎不变 → 函数"平坦"

**R1 惩罚 `‖∇_x D(x)‖² → 0`，就是在说："让 D 在真实样本 x 附近，无论输入怎么微调，输出都几乎不变。"**

### 2.2 对 D 意味着什么

假设真实样本是"数字 7 的手写图"。R1 惩罚效果：

| 不加 R1 的 D | 加 R1 的 D |
|---|---|
| 看到 7 → 输出 0.99；把 7 的某一像素改一点 → 输出 0.1 | 看到 7 → 输出 0.9；改一点 → 输出 0.88；改很多 → 输出 0.6 |
| **对微小扰动极度敏感** | **对微小扰动不敏感** |

**D 从"死记硬背每个像素的精确值"变成"抓住整体形状特征"**——这正是我们想让 D 学到的。

### 2.3 数学上等价于"约束 Lipschitz 常数"

一个函数 `f` 的 **Lipschitz 常数 `L`** 定义为：

$$
\|f(x_1) - f(x_2)\| \le L \cdot \|x_1 - x_2\| \quad \forall x_1, x_2
$$

**若 `‖∇f‖ ≤ L` 处处成立，则 `f` 是 L-Lipschitz 的**（由中值定理）。

R1 惩罚 `‖∇D‖² → 0` 相当于**把 D 的局部 Lipschitz 常数压到极小**——D 变成"1-Lipschitz 甚至 0-Lipschitz 附近"的平滑函数。

> **WGAN-GP 要 `‖∇D‖ → 1`，R1 要 `‖∇D‖ → 0`**——前者是"约束"，后者是"压平"。后者更激进，但在原始 GAN 框架下更稳。

---

## 三、第二层：代码上"惩罚导数"怎么实现（最反直觉的部分）

### 3.1 问题：PyTorch 怎么"更新参数让导数变小"

普通 `loss.backward()` 的机制：

```
forward:  参数 θ ──计算图──> 输出 z ──loss──> L
backward: L.backward() 自动算 ∂L/∂θ，然后用它更新 θ
```

**问题**：R1 里的"导数" `∂D/∂x` **不是 loss 的一部分**，它是 forward 的**中间产物**。如果把它塞进 loss 再 backward，PyTorch 需要算 **`∂(∂D/∂x)/∂θ`**——**二阶导**！

### 3.2 关键机制：`create_graph=True`

```python
grad = torch.autograd.grad(
    logit_real.sum(), real,
    create_graph=True,          # ← 保留 D(x) → grad 的计算路径
)[0]
r1 = (R1_GAMMA / 2) * grad.pow(2).sum(...).mean()
loss = bce + r1
loss.backward()                 # ← 这里会穿到二阶导
```

**默认 `create_graph=False`**：

```
D(x) --autograd.grad--> ∂D/∂x  [计算图在此释放]
                                  ↓
                          只能当"普通数"用，不能再求导
```

**`create_graph=True`**：

```
D(x) --autograd.grad--> ∂D/∂x  [计算图保留！∂D/∂x 本身是可微张量]
                                  ↓
                        grad.pow(2) 可继续求导到 θ
                                  ↓
                        loss.backward() 时算 ∂(grad²)/∂θ
                                  ↑
                            这就是二阶梯度
```

### 3.3 二阶梯度具体算的是什么

展开看数学：

$$
\text{r1} = \frac{\gamma}{2} \Big\|\frac{\partial D(x;\theta)}{\partial x}\Big\|^2
$$

对参数 `θ` 求导：

$$
\frac{\partial \text{r1}}{\partial \theta} = \gamma \cdot \frac{\partial D}{\partial x} \cdot \frac{\partial^2 D}{\partial x \partial \theta}
$$

**这就是为什么叫"二阶"**——它涉及 D 的**混合二阶偏导** `∂²D/∂x∂θ`。PyTorch 的 autograd 通过 `create_graph=True` 的链式法则自动计算。

**代价**：

- **显存**：保留了 `D(x)` 的计算图（原本 backward 后就释放），+50%~100%
- **计算**：反向传播经过两层（一阶 + 二阶），约 1.5~2 倍时间
- **这是 R1 的固有成本**，但换来的是训练稳定，值。

### 3.4 为什么"惩罚导数"能更新参数（直观）

把 `∂D/∂x` 想成"输出对输入的敏感度曲线"，它本身是 **θ 的函数**。

**反向传播的语义**：

- 若某个 `θ` 让 `∂D/∂x` 变大 → 梯度会推 `θ` 往"让 `∂D/∂x` 变小"的方向走
- **就像普通训练让输出接近标签，R1 让导数接近 0**

一句话：**"R1 是优化问题中的一项，梯度下降自然地把它最小化。"**

---

## 四、第三层：动力学上为什么能稳定 GAN（关键动机）

### 4.1 问题诊断：GAN 训练是"双人博弈"，不是"单人优化"

普通训练：`min_θ L(θ)`，梯度下降必然下降（凸情况下）。

**GAN 是双人博弈**：

```
G 想最小化 L_G(θ_G, θ_D)   ← 依赖 D 的参数
D 想最小化 L_D(θ_G, θ_D)   ← 依赖 G 的参数
```

这就像两人在**拔河**——不是找"最低点"，而是找**"纳什均衡"**（谁单独改变策略都不会变好）。

**经典结果**：均衡点附近，梯度下降-上升（GDA）是**不稳定**的——会"绕圈"甚至"发散"。这就是 GAN 早期训练震荡剧烈、模式崩塌的动力学原因（**Goodfellow 2014 论文就提到，后续 Mescheder 2018 严格证明**）。

### 4.2 R1 如何改变动力学

Mescheder 2018 的核心定理（简化版）：

> **原始 GAN 的连续时间动力学在均衡点附近有"纯虚特征值"→ 不收敛（绕圈）。**
>
> **加入 R1 后，特征值变成"有负实部"→ 局部指数收敛。**

**直觉解释**：

- **无 R1**：D 的决策边界"尖刺化"（局部梯度大）→ G 生成的样本稍微不对，D 输出突变 → G 拿到的梯度方向"左右横跳" → 训练震荡
- **有 R1**：D 的边界"平滑"（局部梯度小）→ G 收到的梯度是"连续变化的" → G 能稳步往真实分布移动

### 4.3 为什么"只在真实样本上"做 R1 就够

**理论**：GAN 的目标是让 `p_g → p_data`。**真实样本 `x ~ p_data` 是"目标锚点"**，也是 G 最终要到达的位置。

- 在 `p_data` 附近 R1 把 D 压平 → G 往这个方向走时，D 给出的梯度**平滑**
- 一旦 D 在 `p_data` 附近平滑，神经网络的局部平滑性会**传播到附近区域**（不是全局，但足够）
- 生成样本 `G(z)` 随着训练逐渐靠近 `p_data`，自然也落入"平滑区"

**成本考虑**：

- 真实样本是**固定的**（每个 batch 来自 data loader），计算 R1 成本可预测
- 生成样本**每步都在动**，在其上做 R1 相当于追"移动靶"；R2（在生成样本上做 R1）可与 R1 一起用，但单独 R2 不如 R1 稳

---

## 五、总结：三层机制

| 层面 | 机制 |
|---|---|
| **数学** | `‖∇_x D(x)‖²` 是 D 的局部 Lipschitz 常数平方；惩罚它 → D 在真实样本附近变平 |
| **代码** | `real.requires_grad_(True)` + `autograd.grad(..., create_graph=True)` + `loss.backward()` → 计算并回传**二阶混合偏导 `∂²D/∂x∂θ`** |
| **动力学** | GAN 是双人博弈，均衡点是"鞍点"而非"极小值"；R1 把 D 的 Hession 正定化 → 训练从"绕圈震荡"变成"局部收敛" |

**一句话记住**：**R1 惩罚是"用二阶梯度约束 D 的局部平滑性"**——技术上靠 `create_graph=True` 让"导数"也能被反向传播更新，动力学上把 GAN 的不稳定鞍点变成稳定焦点。